In [ ]:
%load_ext autoreload
%autoreload 2
from datetime import date
from athletics_performance import Athlete, Event, Performance

In [ ]:
# --- Construction via raw scalars ---
p1 = Performance(
    perf_id="P1",
    date=date(2026, 3, 15),
    result_value=12.34,
    measurement="time",
    unit="s",
    athlete_id="2275784",
    event_id="100m",
    category_snapshot="M1M",
    venue="Lyon",
)

# --- Construction via Athlete + Event objects ---
athlete = Athlete(licence="2275784", last_name="Perrin", first_name="Guillaume", yob=1982, sex="M")
event   = Event(event_id="100m", name="100 mètres", measurement="time", unit="s")
p2 = Performance(
    perf_id="P2",
    date=date(2026, 3, 15),
    result_value=12.34,
    measurement="time",
    unit="s",
    athlete=athlete,
    event=event,
    category_snapshot="M1M",
)

print("=== repr ===")
print(p1)
print(p2)

print("\n=== fields ===")
print("perf_id          :", p1.perf_id)
print("athlete_id       :", p1.athlete_id)
print("event_id         :", p1.event_id)
print("date             :", p1.date)
print("result_value     :", p1.result_value)
print("measurement      :", p1.measurement)
print("unit             :", p1.unit)
print("venue            :", p1.venue)
print("category_snapshot:", p1.category_snapshot)

print("\n=== athlete_id from Athlete object ===")
print("p2.athlete_id :", p2.athlete_id)   # derived from athlete.licence
print("p2.event_id   :", p2.event_id)     # derived from event.event_id

In [ ]:
# --- Date formats ---
p_dd = Performance(perf_id="PD1", date="15/03/2026", result_value=1.0, measurement="distance", unit="m", athlete_id="1", event_id="LJ")
p_iso = Performance(perf_id="PD2", date="2026-03-15", result_value=1.0, measurement="distance", unit="m", athlete_id="1", event_id="LJ")
p_obj = Performance(perf_id="PD3", date=date(2026, 3, 15), result_value=1.0, measurement="distance", unit="m", athlete_id="1", event_id="LJ")

print("dd/mm/YYYY :", p_dd.date)
print("YYYY-MM-DD :", p_iso.date)
print("date obj   :", p_obj.date)
print("all equal  :", p_dd.date == p_iso.date == p_obj.date)

print("\n=== invalid date format ===")
try:
    Performance(perf_id="X", date="03-15-2026", result_value=1.0, measurement="distance", unit="m", athlete_id="1", event_id="LJ")
except ValueError as e:
    print(f"Correctly raised ValueError: {e}")

In [ ]:
# --- Result string parsing ---
cases = [
    ("12.34",   "time",     12.34),    # plain seconds
    ("12,34",   "time",     12.34),    # comma decimal
    ("1:23.45", "time",     83.45),    # mm:ss.cc
    ("1:03:05", "time",  3785.0),      # h:mm:ss
    ("6.78",    "distance",  6.78),    # metres
    ("6,78",    "distance",  6.78),    # comma decimal
]

print(f"{'raw':<12} {'meas':<10} {'expected':>10} {'got':>10} {'ok'}")
print("-" * 50)
for raw, meas, expected in cases:
    unit = "s" if meas == "time" else "m"
    p = Performance(perf_id="X", date=date(2026,1,1), result_value=raw, measurement=meas, unit=unit, athlete_id="1", event_id="E")
    ok = "✓" if abs(p.result_value - expected) < 1e-9 else "✗"
    print(f"{raw:<12} {meas:<10} {expected:>10} {p.result_value:>10} {ok}")

In [ ]:
# --- yos property ---
yos_cases = [
    (date(2026, 3, 15),  2026),   # Jan–Aug → same year
    (date(2026, 8, 31),  2026),   # Aug 31 boundary
    (date(2026, 9,  1),  2027),   # Sept 1 → next season
    (date(2025, 10, 1),  2026),   # Oct → season 2026
    (date(2025, 12, 31), 2026),   # Dec 31 → season 2026
]

print(f"{'date':<14} {'expected yos':>13} {'got':>6} {'ok'}")
print("-" * 40)
for d, expected in yos_cases:
    p = Performance(perf_id="X", date=d, result_value=1.0, measurement="distance", unit="m", athlete_id="1", event_id="LJ")
    ok = "✓" if p.yos == expected else "✗"
    print(f"{str(d):<14} {expected:>13} {p.yos:>6} {ok}")

In [ ]:
# --- Optional fields ---
p_full = Performance(
    perf_id="PF",
    date=date(2026, 3, 15),
    result_value=6.78,
    measurement="distance",
    unit="m",
    athlete_id="2275784",
    event_id="LJ",
    venue="Stade de Gerland",
    club_id_snapshot="069001",
    category_snapshot="M1M",
    notes="Wind: +1.2 m/s",
)
print("venue            :", p_full.venue)
print("club_id_snapshot :", p_full.club_id_snapshot)
print("category_snapshot:", p_full.category_snapshot)
print("notes            :", p_full.notes)

# athlete_category alias
p_alias = Performance(perf_id="PA", date=date(2026,3,15), result_value=1.0, measurement="distance", unit="m",
                      athlete_id="1", event_id="LJ", athlete_category="SEF")
print("\nathlete_category alias → category_snapshot:", p_alias.category_snapshot)

In [ ]:
# --- Validation errors ---
tests = [
    dict(measurement="jump", unit="s",  label="bad measurement"),
    dict(measurement="time", unit="kg", label="bad unit"),
]
for kw in tests:
    label = kw.pop("label")
    try:
        Performance(perf_id="X", date=date(2026,1,1), result_value=1.0,
                    athlete_id="1", event_id="E", **kw)
        print(f"{label}: ERROR – should have raised ValueError")
    except ValueError as e:
        print(f"{label}: correctly raised ValueError: {e}")

# bad Athlete type
try:
    Performance(perf_id="X", date=date(2026,1,1), result_value=1.0,
                measurement="time", unit="s", event_id="E", athlete="not_an_athlete")
except TypeError as e:
    print(f"bad athlete type: correctly raised TypeError: {e}")